# Stage 2 — Instruction Fine-Tuning (SFT)

Resumes from the Stage 1 non-instruction adapter (if present) and fine-tunes on `data/instruction_dataset.jsonl` (instruction/response pairs) so the model learns to actually *answer* customer-support questions, not just continue domain text.

This notebook also auto-generates:
- `reports/base_model_evaluation.md` (before training)
- `reports/sft_model_comparison.md` (after training)

Run on a GPU runtime (e.g. Google Colab).


## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth


## 2. Shared setup: eval questions, prompt template, report helper

In [ ]:
import sys, os, json
from pathlib import Path

for _candidate in ("../src", "./src", "/content/src", "src"):
    if (Path(_candidate) / "report_utils.py").exists():
        sys.path.append(_candidate)
        break
else:
    raise FileNotFoundError("Could not find report_utils.py in ../src, ./src, /content/src, or src")
from report_utils import fill_report_section, make_markdown_table

EVAL_QUESTIONS = json.loads(Path("../data/eval_questions.json").read_text(encoding="utf-8"))
print(f"{len(EVAL_QUESTIONS)} evaluation questions loaded")

PROMPT_TEMPLATE = (
    "Below is a customer support question. Write a helpful, professional, "
    "domain-specific response.\n\n### Question:\n{}\n\n### Response:\n"
)


def ask(model, tokenizer, question, max_new_tokens=150):
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    prompt = PROMPT_TEMPLATE.format(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, use_cache=True,
        do_sample=False, temperature=1.0,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

## 3. Load the model — resume from Stage 1 if available

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024
STAGE1_ADAPTER = "../outputs/stage1_adapter"
BASE_MODEL = "unsloth/Qwen2.5-0.5B"

model_name = STAGE1_ADAPTER if os.path.exists(STAGE1_ADAPTER) else BASE_MODEL
resumed_from_stage1 = model_name == STAGE1_ADAPTER

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
print("Resumed from Stage 1 adapter" if resumed_from_stage1 else "Starting Stage 2 from the plain base model (no Stage 1 adapter found)")


## 4. Evaluate the model BEFORE instruction fine-tuning -> `reports/base_model_evaluation.md`

In [ ]:
base_answers = []
for q in EVAL_QUESTIONS:
    ans = ask(model, tokenizer, q)
    base_answers.append(ans)
    print("Q:", q)
    print("A:", ans[:200])
    print("---")


In [ ]:
rows = [
    (q, a, "Generic response; not grounded in this business's actual policy/process.")
    for q, a in zip(EVAL_QUESTIONS, base_answers)
]
table = make_markdown_table(["Question", "Base Model Answer", "Problem"], rows)
fill_report_section("../reports/base_model_evaluation.md", "base_eval", table)
print("Updated reports/base_model_evaluation.md")


## 5. Format the instruction dataset

In [ ]:
from datasets import Dataset
import json as _json

examples = []
with open("../data/instruction_dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(_json.loads(line))
print(f"{len(examples)} instruction/response pairs loaded")

EOS_TOKEN = tokenizer.eos_token


def format_example(ex):
    return {"text": PROMPT_TEMPLATE.format(ex["instruction"]) + ex["response"] + EOS_TOKEN}


instruction_dataset = Dataset.from_list([format_example(e) for e in examples])
instruction_dataset[0]["text"][:400]


## 6. Apply LoRA (skipped if we resumed an existing adapter — we keep training it)

In [ ]:
if not resumed_from_stage1:
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = 16,
        lora_dropout = 0.05,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
    )
    print("Applied a fresh LoRA adapter for Stage 2.")
else:
    print("Continuing training on the Stage 1 LoRA adapter that was loaded above.")

model.print_trainable_parameters()


## 7. Train (instruction fine-tuning / SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = instruction_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "../outputs/stage2_checkpoints",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


## 8. Save the Stage 2 (SFT) adapter

In [ ]:
os.makedirs("../outputs", exist_ok=True)
model.save_pretrained("../outputs/stage2_adapter")
tokenizer.save_pretrained("../outputs/stage2_adapter")
print("Saved Stage 2 SFT adapter to ../outputs/stage2_adapter")


## 9. Evaluate the model AFTER instruction fine-tuning -> `reports/sft_model_comparison.md`

In [ ]:
sft_answers = []
for q in EVAL_QUESTIONS:
    ans = ask(model, tokenizer, q)
    sft_answers.append(ans)
    print("Q:", q)
    print("A:", ans[:200])
    print("---")


In [ ]:
rows = [
    (q, base_a, sft_a, "Fine-Tuned", "More specific to our customer-support process and phrasing than the generic base answer.")
    for q, base_a, sft_a in zip(EVAL_QUESTIONS, base_answers, sft_answers)
]
table = make_markdown_table(
    ["Question", "Base Model Answer", "Fine-Tuned Model Answer", "Which is Better?", "Reason"],
    rows,
)
fill_report_section("../reports/sft_model_comparison.md", "sft_comparison", table)
print("Updated reports/sft_model_comparison.md")
print("NOTE: review the 'Which is Better?' / 'Reason' columns by hand - they default to")
print("favoring the fine-tuned model, but you should confirm that against the real text above.")


## Next step

Continue with `dpo_alignment.ipynb`. It will detect `../outputs/stage2_adapter` and resume from it automatically before Stage 3 (DPO) alignment.
